In [1]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os

In [2]:
PROCESSED_PATH = "data/02-processed/birthdays_processed.csv"

assert os.path.exists(PROCESSED_PATH), f"Processed file not found at {PROCESSED_PATH}"

df = pd.read_csv(PROCESSED_PATH)
df["onehot_homicide"] = (df["manner"] == "h").apply(lambda x: int(x))

In [3]:
# Tinkered a lot with the regression, removed elements that didn't seem to help:
# normalized_age = (df['age_floor'] - df['age_floor'].mean()) / df['age_floor'].std()
# subset["age_floor"] = normalized_age

"""
THIS CELL IS ALSO IN THE MAIN IPYNB
It's here also so this notebook can have all the regression and generate the subset dataset

Created a subset of just columns being used in logistic regression, and saved it to use later
Used statsmodels logit function to create a regression that controls for multiple independent variables where some are categorical
For gender, we set the baseline category as male, and we set the baseline category as married for marital status
"""
subset = df[["sex", "onehot_homicide", "marital", "age_floor"]].copy()
subset["marital"] = subset["marital"].apply(lambda x: "d" if x == "w" else x)
regression = smf.logit("onehot_homicide ~ age_floor + C(sex, Treatment(reference='m')) + C(marital, Treatment(reference='m'))", data=subset)
result = regression.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.014886
         Iterations 12
                           Logit Regression Results                           
Dep. Variable:        onehot_homicide   No. Observations:              1939219
Model:                          Logit   Df Residuals:                  1939213
Method:                           MLE   Df Model:                            5
Date:                Thu, 19 Mar 2026   Pseudo R-squ.:                  0.3185
Time:                        07:49:06   Log-Likelihood:                -28868.
converged:                       True   LL-Null:                       -42363.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------
Intercept                                    

In [4]:
"""
Creating a regression for our onehot homicide variable and all possible combinations of our three predictors (except all three because that's what our original regression was)
Then saving results and extracting r-squares for further investigation
"""
age_result = smf.logit("onehot_homicide ~ age_floor", data=subset).fit()
sex_result = smf.logit("onehot_homicide ~ C(sex, Treatment(reference='m'))", data=subset).fit()
marital_result = smf.logit("onehot_homicide ~ C(marital, Treatment(reference='m'))", data=subset).fit()
age_sex_result = smf.logit("onehot_homicide ~ age_floor + C(sex, Treatment(reference='m'))", data=subset).fit()
age_marital_result = smf.logit("onehot_homicide ~ age_floor + C(marital, Treatment(reference='m'))", data=subset).fit()
marital_sex_result = smf.logit("onehot_homicide ~ C(marital, Treatment(reference='m')) + C(sex, Treatment(reference='m'))", data=subset).fit()

results = [age_result, sex_result, marital_result, age_sex_result, age_marital_result, marital_sex_result]
names = ["Just age:", "Just sex:", "Just marital status:", "Age and sex:", "Age and marital status:", "Marital status and sex:"] 
squaredrs = []

for result in results:
    squaredrs.append(result.prsquared)
    print(result.summary())

Optimization terminated successfully.
         Current function value: 0.015266
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.021518
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.019069
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.015020
         Iterations 11
Optimization terminated successfully.
         Current function value: 0.015032
         Iterations 12
Optimization terminated successfully.
         Current function value: 0.018872
         Iterations 11
                           Logit Regression Results                           
Dep. Variable:        onehot_homicide   No. Observations:              1939219
Model:                          Logit   Df Residuals:                  1939217
Method:                           MLE   Df Model:                            1
Date:                Thu, 19 Mar 2026   Pseudo R-squ.:            

In [5]:
# R-squares for each regression
for i in range(len(names)):
    print(names[i])
    print(squaredrs[i])

Just age:
0.3011739043983941
Just sex:
0.014983782381689736
Just marital status:
0.1270741598360886
Age and sex:
0.3124533227832901
Age and marital status:
0.3118908673192038
Marital status and sex:
0.13611873523223272


In [6]:
# Turning R-squares into a dataframe so we can use it in a different notebook
rsquares = pd.DataFrame(data = {"Model": names, "R-squared": squaredrs})
rsquares

,Model,R-squared
0,Just age:,0.301174
1,Just sex:,0.014984
2,Just marital status:,0.127074
3,Age and sex:,0.312453
4,Age and marital status:,0.311891
5,Marital status and sex:,0.136119


In [7]:
# Saving R-squares
rsquares.to_csv("data/02-processed/rsquares.csv", index=False)